In [23]:
import pandas as pd
import sqlalchemy as sal
from datetime import date
import pyodbc

engine = sal.create_engine(r"mssql+pyodbc://AKSHAYKUMAR\SQLEXPRESS01/master?"r"driver=ODBC+DRIVER+18+FOR+SQL+SERVER&"r"TrustServerCertificate=yes")
conn = engine.connect()

In [36]:
def extract():
    df_incoming = pd.read_csv('products.csv')
    df_dim = pd.read_sql_query("SELECT product_id, product_name, price FROM product_dim3", conn)
    return df_incoming, df_dim

def transform(df_incoming, df_dim):
    df_merged = pd.merge(
        df_incoming, 
        df_dim, 
        on='product_id', 
        how='left', 
        suffixes=('_new', '_dim')
    )
    
    new_records = df_merged[df_merged['price_dim'].isna()].copy()
    df_inserts = pd.DataFrame({
        'product_id': new_records['product_id'],
        'product_name': new_records['product_name_new'] if 'product_name_new' in new_records.columns else new_records['product_name'],
        'price': new_records['price_new'],
        'price': new_records['price_new'],
        'previous_price': None,
        'effective_date': pd.Timestamp.now().date()
    })
    
    changed_records = df_merged[
        df_merged['price_dim'].notna() & 
        (df_merged['price_new'] != df_merged['price_dim'])
    ].copy()
    
    df_updates = pd.DataFrame({
        'product_id': changed_records['product_id'],
        'product_name': changed_records['product_name_new'] if 'product_name_new' in changed_records.columns else changed_records['product_name'],
        'price': changed_records['price_new'],
        'previous_price': changed_records['price_dim'],
        'effective_date': pd.Timestamp.now().date()
    })
    
    return df_inserts, df_updates

def load_inserts(df_inserts, conn):
    if not df_inserts.empty:
        df_inserts.to_sql('product_dim3', con=conn, index=False, if_exists='append')
        conn.commit()

def load_updates(df_updates, conn):
    if not df_updates.empty:
        # Parameterized query to prevent SQL injection
        query = sal.text("""
            UPDATE product_dim3
            SET previous_price = :previous_price,
                product_name = :product_name,
                price = :price,
                effective_date = :effective_date
            WHERE product_id = :product_id
        """)
        conn.execute(query, df_updates.to_dict(orient='records'))
        conn.commit()

In [37]:
df_incoming, df_dim = extract()

In [38]:
df_inserts, df_updates = transform(df_incoming, df_dim)

In [39]:
load_inserts(df_inserts, conn)
load_updates(df_updates, conn)